## Loading the data

In [4]:
import pandas as pd
import pandas as pd

raw = pd.read_excel("/Users/macbookpro/Documents/agri_merged_raw.xlsx")
df = raw.copy()

print(df.shape)
df.head()

(25432, 14)


,record_id,observation_date,year,country,iso3,region,crop,plot_area_ha,production_tonnes,yield_kg_ha,yield_hg_ha,rainfall_mm,pesticides_tonnes,temperature_c
0,1,1990-01-01,1990,Afghanistan,AFG,Southern Asia,Maize (corn),273000.0,480000.0,1758.2,NaN,NaN,NaN,NaN
1,2,1991-01-01,1991,Afghanistan,AFG,Southern Asia,Maize (corn),250000.0,420000.0,1680.0,NaN,NaN,NaN,NaN
2,3,1992-01-01,1992,Afghanistan,AFG,Southern Asia,Maize (corn),200000.0,300000.0,1500.0,NaN,NaN,NaN,NaN
3,4,1993-01-01,1993,Afghanistan,AFG,Southern Asia,Maize (corn),280000.0,470000.0,1678.6,NaN,NaN,NaN,NaN
4,5,1994-01-01,1994,Afghanistan,AFG,Southern Asia,Maize (corn),300000.0,500000.0,1666.7,NaN,NaN,NaN,NaN


## Snapshot before cleaning

In [5]:
before_rows = len(raw)
before_columns = raw.shape[1]
before_nulls = raw.isna().sum().sum()
before_duplicates = raw.duplicated().sum()
before_zero_area = (raw["plot_area_ha"] == 0).sum()
before_zero_production = (raw["production_tonnes"] == 0).sum()
before_no_weather = raw["rainfall_mm"].isna().sum()

print("Rows:", before_rows)
print("Columns:", before_columns)
print("Missing cells:", before_nulls)
print("Duplicate rows:", before_duplicates)
print("Zero area rows:", before_zero_area)
print("Zero production rows:", before_zero_production)
print("Rows with missing rainfall:", before_no_weather)

Rows: 25432
Columns: 14
Missing cells: 49527
Duplicate rows: 0
Zero area rows: 242
Zero production rows: 213
Rows with missing rainfall: 12328


## Count of missing values in each column

In [6]:
missing = raw.isna().sum()
missing_pct = (missing / len(raw) * 100).round(1)

pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
record_id,0,0.0
observation_date,0,0.0
year,0,0.0
country,0,0.0
iso3,0,0.0
region,0,0.0
crop,0,0.0
plot_area_ha,2,0.0
production_tonnes,0,0.0
yield_kg_ha,213,0.8


## Tidy the text columns

In [7]:
text_columns = ["country", "iso3", "region", "crop"]

for col in text_columns:
    df[col] = df[col].str.strip()

spacing_fixes = 0

for col in text_columns:
    spacing_fixes = spacing_fixes + (df[col] != raw[col]).sum()

print("Cells fixed:", spacing_fixes)

Cells fixed: 0


## Check the iso3 country codes

In [8]:
df["iso3"] = df["iso3"].str.upper()
not_a_code = df["iso3"].str.len() != 3

print("Rows with a bad iso3:", not_a_code.sum())
print(raw.loc[not_a_code, "iso3"].value_counts())

iso3_fixed = not_a_code.sum()
df.loc[not_a_code, "iso3"] = pd.NA

Rows with a bad iso3: 134
iso3
Serbia and Montenegro    70
Belgium-Luxembourg       40
USSR                     12
Yugoslav SFR             12
Name: count, dtype: int64


## Fix dates

In [9]:
df["observation_date"] = pd.to_datetime(df["observation_date"])

date_year = df["observation_date"].dt.year
year_mismatch = (date_year != df["year"]).sum()
print("Rows where date and year disagree:", year_mismatch)

df.dtypes

Rows where date and year disagree: 0


record_id                     int64
observation_date     datetime64[us]
year                          int64
country                         str
iso3                            str
region                          str
crop                            str
plot_area_ha                float64
production_tonnes           float64
yield_kg_ha                 float64
yield_hg_ha                 float64
rainfall_mm                 float64
pesticides_tonnes           float64
temperature_c               float64
dtype: object

## Check for duplicate rows

In [10]:
duplicates_found = df.duplicated(subset=["country", "crop", "year"]).sum()
print("Duplicates found:", duplicates_found)

df = df.drop_duplicates(subset=["country", "crop", "year"])
print("Rows after removing duplicates:", len(df))

Duplicates found: 0
Rows after removing duplicates: 25432


## Check for impossible values(Negative values)

In [11]:
number_columns = ["plot_area_ha", "production_tonnes", "yield_kg_ha",
                  "rainfall_mm", "pesticides_tonnes"]

for col in number_columns:
    negatives = (df[col] < 0).sum()
    print(col, "->", negatives, "negative values")

plot_area_ha -> 0 negative values
production_tonnes -> 0 negative values
yield_kg_ha -> 0 negative values
rainfall_mm -> 0 negative values
pesticides_tonnes -> 0 negative values


## Add flags to the data(Mark rows instead of deleting them)

In [12]:
former_states = [
    "Czechoslovakia", "Ethiopia PDR", "Sudan (former)", "USSR",
    "Yugoslav SFR", "Serbia and Montenegro", "Belgium-Luxembourg"
]

df["has_weather"] = df["rainfall_mm"].notna()
df["valid_yield"] = df["yield_kg_ha"] > 0
df["is_former_entity"] = df["country"].isin(former_states)
df["area_zero_but_production"] = (
    (df["plot_area_ha"] == 0) & (df["production_tonnes"] > 0)
)

kg_from_hg = df["yield_hg_ha"] / 10
gap = (kg_from_hg - df["yield_kg_ha"]).abs() / df["yield_kg_ha"]
df["yield_source_mismatch"] = (
    df["yield_hg_ha"].notna() & (df["yield_kg_ha"] > 0) & (gap > 0.01)
)

flag_columns = [
    "has_weather", "valid_yield", "is_former_entity",
    "area_zero_but_production", "yield_source_mismatch"
]
df[flag_columns].sum()

has_weather                 13104
valid_yield                 25218
is_former_entity              346
area_zero_but_production       32
yield_source_mismatch         712
dtype: int64

## Flag outliers using IQR rule

In [15]:
## Part A. Check yield and production within each crop

df["yield_outlier"] = False
df["production_outlier"] = False

for crop in df["crop"].unique():
    for col, flag in [("yield_kg_ha", "yield_outlier"),
                      ("production_tonnes", "production_outlier")]:
        values = df.loc[df["crop"] == crop, col]
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        is_outlier = (values < low) | (values > high)
        df.loc[is_outlier[is_outlier].index, flag] = True

## Part B. Check pesticides and rainfall across all rows

df["pesticide_outlier"] = False
df["rainfall_outlier"] = False

for col, flag in [("pesticides_tonnes", "pesticide_outlier"), ("rainfall_mm", "rainfall_outlier")]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    df[flag] = (df[col] < low) | (df[col] > high)

outlier_columns = ["yield_outlier", "production_outlier", "pesticide_outlier", "rainfall_outlier"]
df[outlier_columns].sum()

yield_outlier         1012
production_outlier    3724
pesticide_outlier     1916
rainfall_outlier       207
dtype: int64

## Add two useful columns

In [16]:
df["yield_t_ha"] = (df["yield_kg_ha"] / 1000).round(3)

df["period"] = pd.cut(
    df["year"],
    bins=[1989, 1995, 2001, 2007, 2013],
    labels=["1990-95", "1996-2001", "2002-07", "2008-13"]
)

df["period"].value_counts().sort_index()

period
1990-95      6200
1996-2001    6395
2002-07      6434
2008-13      6403
Name: count, dtype: int64

## Check data set size and save

In [18]:
rows = len(df)
columns = df.shape[1]

print("Cleaned shape:", rows, "rows x", columns, "columns")

if rows >= 15000 and columns >= 10:
    print("PASS: at least 15,000 rows and 10 columns")

    output_file = "/Users/macbookpro/Documents/agri_cleaned.csv"
    df.to_csv(output_file, index=False)
    print("Saved to:", output_file)
else:
    print("FAIL: revise the cleaning method")

Cleaned shape: 25432 rows x 25 columns
PASS: at least 15,000 rows and 10 columns
Saved to: /Users/macbookpro/Documents/agri_cleaned.csv


## Compare the data before and after cleaning

In [19]:
## Compare the counts
summary = pd.DataFrame({
    "measure": [
        "Rows", "Columns", "Missing cells (original 14 columns)",
        "Duplicate rows", "Zero area rows", "Zero production rows",
        "Rows with no weather"
    ],
    "before": [
        before_rows, before_columns, before_nulls, before_duplicates,
        before_zero_area, before_zero_production, before_no_weather
    ],
    "after": [
        len(df), df.shape[1], df[raw.columns].isna().sum().sum(),
        df.duplicated().sum(), (df["plot_area_ha"] == 0).sum(),
        (df["production_tonnes"] == 0).sum(), df["rainfall_mm"].isna().sum()
    ],
})

summary

,measure,before,after
0,Rows,25432,25432
1,Columns,14,25
2,Missing cells (original 14 columns),49527,49661
3,Duplicate rows,0,0
4,Zero area rows,242,242
5,Zero production rows,213,213
6,Rows with no weather,12328,12328


## What was changed

In [20]:
changes = pd.DataFrame({
    "what we did": [
        "Spacing fixes in text",
        "iso3 codes set to missing (were country names)",
        "Duplicate records removed",
        "Flag columns added",
        "Outlier columns added",
        "Rows deleted"
    ],
    "count": [
        spacing_fixes, iso3_fixed, duplicates_found,
        5, 4, before_rows - len(df)
    ],
})

changes

,what we did,count
0,Spacing fixes in text,0
1,iso3 codes set to missing (were country names),134
2,Duplicate records removed,0
3,Flag columns added,5
4,Outlier columns added,4
5,Rows deleted,0


### Cleaning Summary

We kept all **25,432 rows**. The dataset went from **14 to 25 columns** after we added flags, converted yield to tonnes per hectare, and grouped the years into periods.

We found **no duplicate records**, **no date and year mismatches**, and **no negative values in the five measurement columns we checked**. The spacing check made no changes.

We set **134 historical labels** in the `iso3` column to missing because they fell outside the three-character format. This increased the number of missing cells in the original 14 columns from **49,527 to 49,661**. The country names and rows were kept.

We left missing measurements blank and flagged unusual values for review. The cleaned dataset passed the minimum size check and was saved as `agri_cleaned.csv`.

## Q1. Which crops had the highest median yield per hectare?

In [21]:
good_area = df["plot_area_ha"] > 0
good_yield = df["yield_kg_ha"] >= 0
yield_data = df[good_area & good_yield]

crop_yield = yield_data.groupby("crop")["yield_kg_ha"]
crop_table = crop_yield.agg(["count", "mean", "median", "min", "max"])

crop_table.sort_values("median", ascending=False).round(2)

,count,mean,median,min,max
crop,,,,,
Potatoes,3634,17191.14,15224.20,840.6,52229.1
"Cassava, fresh",2345,10254.10,9612.10,1091.4,43667.9
Plantains and cooking bananas,1147,9990.60,8156.00,28.6,55485.5
Yams,1304,8482.73,7659.30,136.7,33333.3
Sweet potatoes,2636,9362.09,7556.85,221.1,40000.0
Rice,2753,3396.24,3064.60,285.7,10389.5
Wheat,2909,2819.60,2313.10,131.4,9938.7
Maize (corn),3804,3622.90,2273.35,84.9,36761.9
Soya beans,2154,1540.31,1466.70,5.0,4160.9


#### Filtered for rows with positive area and a recorded, nonnegative yield. Then I grouped by crop, calculated five statistics, and sorted by median yield from highest to lowest.”

#### Interpretation: Potatoes had the highest median yield at 15,224.20 kg/ha, followed by fresh cassava at 9,612.10 kg/ha. These medians describe typical yields per hectare for the records available for each crop.

## Q2. How did median maize yield vary across regions?

In [22]:
maize = yield_data[yield_data["crop"] == "Maize (corn)"]

region_counts = maize["region"].value_counts()
maize_table = maize.groupby("region")["yield_kg_ha"].median().to_frame()
maize_table["records"] = region_counts

maize_table = maize_table[maize_table["records"] >= 20]
maize_table.sort_values("yield_kg_ha", ascending=False).round(2)

,yield_kg_ha,records
region,,
Western Europe,8982.90,148
Northern America,8312.45,48
Australia and New Zealand,7059.90,48
Southern Europe,5814.05,224
Eastern Asia,4555.50,120
Western Asia,4550.40,330
Eastern Europe,4119.50,229
Central Asia,3807.50,110
Northern Europe,2606.05,26


#### Selected maize records, used value_counts() to count records in each region, and calculated each region’s median. I displayed regions with at least 20 records.”

#### Interpretation: Western Europe had the highest median among the regions shown at 8,982.90 kg/ha. Eastern Africa had a median of 1,506.20 kg/ha across 400 records.

## Q3. How did recorded rainfall vary across regions?

In [23]:
rain = df.dropna(subset=["rainfall_mm"])
rain = rain.drop_duplicates(subset=["region", "country"])

rain_table = rain.groupby("region")["rainfall_mm"].agg(["count", "median"])
rain_table.sort_values("median", ascending=False).round(2)

,count,median
region,,
Melanesia,1,3142.0
South-eastern Asia,3,2702.0
Central America,5,1976.0
South America,9,1761.0
Eastern Asia,1,1668.0
Southern Asia,5,1500.0
Caribbean,4,1425.0
Middle Africa,3,1342.0
Eastern Africa,11,1180.0


#### Kept rows with rainfall and used one reading per country or historical entity in each region. The recorded rainfall value repeats across crop and year rows, so counting it once gives each entity equal weight.”

#### Interpretation: Melanesia had the highest recorded median rainfall at 3,142 mm, based on a single entity with rainfall data. The regional medians describe only the entities with recorded rainfall.

## Q4. How did Kenya’s median maize yield change between 1990–2001 and 2002–2013?

In [24]:
kenya_maize = maize[maize["iso3"] == "KEN"].copy()

kenya_maize["period"] = "1990–2001"
later_years = kenya_maize["observation_date"].dt.year >= 2002
kenya_maize.loc[later_years, "period"] = "2002–2013"

period_table = kenya_maize.groupby("period")["yield_kg_ha"].agg(
    ["count", "median"]
)
period_table.round(2)

,count,median
period,,
1990–2001,12,1664.40
2002–2013,12,1666.35


#### Selected Kenya’s maize records and used the year from observation_date to split the 1990–2013 data into two periods. Then I calculated the median for each period.”

#### Interpretation: Kenya’s median maize yield rose from 1,664.40 kg/ha to 1,666.35 kg/ha, an increase of 1.95 kg/ha. Each period had 12 annual records, and the median changed very little.

## Q5. How did Kenya’s maize yield differ between cooler and warmer years?

In [25]:
kenya_temp = kenya_maize.dropna(subset=["temperature_c"]).copy()
median_temp = kenya_temp["temperature_c"].median()

kenya_temp["temperature_group"] = "Cooler"
warmer_years = kenya_temp["temperature_c"] > median_temp
kenya_temp.loc[warmer_years, "temperature_group"] = "Warmer"

print("Median temperature:", round(median_temp, 2), "°C")

temp_table = kenya_temp.groupby("temperature_group")["yield_kg_ha"].agg(
    ["count", "median"]
)
temp_table.round(2)

Median temperature: 16.66 °C


,count,median
temperature_group,,
Cooler,12,1685.45
Warmer,11,1640.50


#### Kept the years with temperature readings. I called years above the median temperature ‘warmer’ and years at or below it ‘cooler,’ then compared their median yields.”

#### Interpretation: At or below 16.66 °C, the median maize yield was 1,685.45 kg/ha across 12 years. Above that temperature, the median was 1,640.50 kg/ha across 11 years. This shows a difference between the two groups, and other factors such as farm inputs could also affect yield.